# DynaPool reviewer-revision experiments
This notebook stores every completed epoch in Google Drive. After a Colab disconnect, reconnect and rerun the setup, data, and current training cells; completed runs are skipped and the active run resumes from its last checkpoint.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/youngsilver-kim/DynaPool-Analysis.git'
REPO_DIR = '/content/DynaPool-Analysis'
DRIVE_OUTPUT = '/content/drive/MyDrive/DynaPool_Reviewer_Revision'
EXPERIMENT = 'reviewer_revision_v1'

import os, subprocess
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
print('Repository:', os.getcwd())
print('Persistent output:', DRIVE_OUTPUT)

In [ ]:
!python -m pip install -q -r requirements.txt
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY — select a GPU runtime')

## Data setup
Tiny-ImageNet is kept on Colab's local disk for speed. It must be downloaded again after a new runtime starts; experimental checkpoints remain in Drive.

In [ ]:
!python download_data.py --data-dir /content/data --delete-archive

## Optional two-epoch smoke test
This tests code and checkpointing only. FakeData results must not appear in the paper.

In [ ]:
!python train.py --fake-data --methods avg dyna --seeds 13 --epochs 2 --warmup-epochs 1 --batch-size 32 --eval-batch-size 64 --num-workers 2 --output-root /content/smoke_outputs --experiment-name smoke_test

## Phase 1 — five primary methods × three seeds
Rerun this same cell after any disconnect. It resumes the interrupted seed/method and skips completed runs.

In [ ]:
!python train.py --suite baselines --seeds 13 42 2026 --epochs 50 --warmup-epochs 5 --batch-size 128 --eval-batch-size 256 --num-workers 2 --data-root /content/data/tiny-imagenet-200 --output-root "{DRIVE_OUTPUT}" --experiment-name "{EXPERIMENT}"

## Phase 2 — reviewer-required ablations
Start only after Phase 1 is complete. The static-mean control reads the same-seed DynaPool training-set alpha mean. This cell is also auto-resumable.

In [ ]:
!python train.py --suite reviewer --seeds 13 42 2026 --epochs 50 --warmup-epochs 5 --batch-size 128 --eval-batch-size 256 --num-workers 2 --data-root /content/data/tiny-imagenet-200 --output-root "{DRIVE_OUTPUT}" --experiment-name "{EXPERIMENT}"

## Reviewer-facing statistics and figures

In [ ]:
EXPERIMENT_DIR = f'{DRIVE_OUTPUT}/{EXPERIMENT}'
!python analyze.py --experiment-dir "{EXPERIMENT_DIR}"

## Efficiency measurement
Run all five methods in this one cell on the same GPU. Keep its environment JSON with the CSV files.

In [ ]:
!python measure_efficiency.py --methods avg max gem att dyna --batch-size 1 --warmup 100 --repetitions 300 --trials 10 --output-dir "{EXPERIMENT_DIR}/efficiency"

## Final audit
Open `REVIEWER_CHECKLIST.md` and check every item against the saved outputs before changing the manuscript's claims. Three seeds are the requested minimum; if compute permits, add seeds 7 and 101 to both phases using the exact same settings.

In [ ]:
!python audit_experiment.py --experiment-dir "{EXPERIMENT_DIR}" --seeds 13 42 2026 --require-efficiency